**Installing the python SDK wikirate4py**

In [ ]:
pip install wikirate4py

**Connecting to wikirate API**

In [ ]:
import wikirate4py
from wikirate4py.utils import to_dataframe

api = wikirate4py.API("yexBJaUij2HvTrsUILvvJgtt")
print(api.get_metrics(limit=1))

[{'id': 826615, 'name': 'Direct greenhouse gas (GHG) emissions (Scope 1), GRI 305-1-a (formerly G4-EN15-a)', 'designer': 'Global Reporting Initiative', 'question': 'What is the amount of greenhouse gas (GHG) emissions (in tonnes of CO2 equivalent) that the organization is directly responsible for?', 'metric_type': 'Researched', 'about': 'This metric is based on the Global Reporting Initiative (GRI) Standard\nGuidelines.\n\nIn 2018, the GRI G4 Sustainability Reporting Guidelines were superseded by the\nGRI Sustainability Reporting Standards (GRI standards). For this metric, the\ncode G4-EN15-a is used in company reporting prior to 2018, and the new GRI\n305-1 code used in reporting from 2018 onwards. The methodology for the former\nG4 standard for this metric can be found\n[here](/Direct_greenhouse_gas_GHG_emissions_Scope_1_G4_EN15_a_Methodology).\n\nIn the context of the GRI Standards, the environmental dimension of\nsustainability concerns an organization’s impacts on living and non-l

**Declaring the required Metrics**

- Importing the required Libraries and Decalring the wikirate API key to connect with wikirate data
- Using the wikirate4py library to interact with the wikirate API

In [ ]:
import wikirate4py
import time
import re
from collections import defaultdict
from wikirate4py.exceptions import TooManyRequestsException

API_KEY = "yexBJaUij2HvTrsUILvvJgtt"
api = wikirate4py.API(API_KEY)

METRICS = [
    {
        "pillar": "E",
        "metric_designer": "Global Reporting Initiative",
        "metric_name": "Direct greenhouse gas (GHG) emissions (Scope 1), GRI 305-1-a (formerly G4-EN15-a)",
        "question": "What is the total amount of Scope 1 direct greenhouse gas emissions in metric tonnes of CO2 equivalent?",
        "custom_id": "GRI_305_1_a_Scope1_GHG",
        "unit": "tCO2e"
    },
    {
        "pillar": "S",
        "metric_designer": "Global Reporting Initiative",
        "metric_name": "Employees, GRI 102-8-a (formerly G4-10-a)",
        "question": "What is the company's total number of employees?",
        "custom_id": "GRI_102_8_a_Total_Employees",
        "unit": "employees"
    },
    {
        "pillar": "G",
        "metric_designer": "Global Reporting Initiative",
        "metric_name": "Female employees, GRI 102-8-a (formerly G4-10-a)",
        "question": "What is the total number of female employees?",
        "custom_id": "GRI_102_8_a_Female_Employees",
        "unit": "employees"
    }
]

YEARS = [2022, 2023]
MAX_RECORDS_PER_METRIC_YEAR = 15
SLEEP_BETWEEN_PAGES = 0.3
SLEEP_BETWEEN_METRIC_YEARS = 1.0

print("Years selected:", YEARS)
print("Metrics selected:", [m["custom_id"] for m in METRICS])
print("Max records per metric-year:", MAX_RECORDS_PER_METRIC_YEAR)

Years selected: [2022, 2023]
Metrics selected: ['GRI_305_1_a_Scope1_GHG', 'GRI_102_8_a_Total_Employees', 'GRI_102_8_a_Female_Employees']
Max records per metric-year: 15


**2**. **Fetching Data from WikiRate**

- WikiRate exposes a RESTful card-based API where every entity which contains Company, Metric, Answer, Source.
- Defining ***fetch_raw_answers_for_metric_year*** to extract raw data from wikirate

In [ ]:
def fetch_raw_answers_for_metric_year(metric: dict, year: int, max_records: int) -> list:
    answers = []

    cursor = wikirate4py.Cursor(
        api.get_answers,
        metric_name=metric["metric_name"],
        metric_designer=metric["metric_designer"],
        year=year,
        per_page=100
    )

    while cursor.has_next() and len(answers) < max_records:
        batch = cursor.next()
        answers.extend(batch[: max_records - len(answers)])
        time.sleep(SLEEP_BETWEEN_PAGES)

    return answers

raw_answers = defaultdict(dict)

for metric in METRICS:
    for year in YEARS:
        print(f"Fetching raw: {metric['custom_id']} | {year}")
        batch = fetch_raw_answers_for_metric_year(
            metric=metric,
            year=year,
            max_records=MAX_RECORDS_PER_METRIC_YEAR
        )
        raw_answers[metric["custom_id"]][year] = batch
        print(batch)
        print(f"  Raw answers fetched: {len(batch)}")
        time.sleep(SLEEP_BETWEEN_METRIC_YEARS)

Fetching raw: GRI_305_1_a_Scope1_GHG | 2022
[{'id': 7174977, 'metric': 'Global Reporting Initiative+Direct greenhouse gas (GHG) emissions (Scope 1), GRI 305-1-a (formerly G4-EN15-a)', 'company': 'Origin Energy', 'value': '22400000', 'year': 2022, 'comments': 'Page13 in 2020 annual report, P20 in 2020 sustainability report Chrisz[https://wikirate.org/Chrisz].....2021-04-12 07:20:52 UTC According to p18 graph 5, 11/13*18648=15626.7692 Chrisz[https://wikirate.org/Chrisz].....2021-04-13 04:08:13 UTC', 'sources': ['Source-000104185'], 'checked_by': None, 'check_requested': None, 'url': 'https://wikirate.org/Global_Reporting_Initiative+Direct_greenhouse_gas_GHG_emissions_Scope_1_GRI_305_1_a_formerly_G4_EN15_a+Origin_Energy+2022'}, {'id': 11520678, 'metric': 'Global Reporting Initiative+Direct greenhouse gas (GHG) emissions (Scope 1), GRI 305-1-a (formerly G4-EN15-a)', 'company': 'Marcus & Millichap Inc.', 'value': 'Unknown', 'year': 2022, 'comments': 'No se encuentran resultados Raúl Román S

In [ ]:
for metric in METRICS:
    metric_id = metric["custom_id"]
    for year in YEARS:
        print(metric_id, year, len(raw_answers[metric_id][year]))

GRI_305_1_a_Scope1_GHG 2022 15
GRI_305_1_a_Scope1_GHG 2023 15
GRI_102_8_a_Total_Employees 2022 15
GRI_102_8_a_Total_Employees 2023 15
GRI_102_8_a_Female_Employees 2022 15
GRI_102_8_a_Female_Employees 2023 15


**Viewing the Raw data format from wikirate**

In [ ]:
def inspect_object(obj, name="object", max_depth=2, _depth=0):
    indent = "  " * _depth

    print(f"{indent}🔎 {name}")
    print(f"{indent}Type: {type(obj)}")

    # Base case
    if _depth >= max_depth:
        print(f"{indent}... (max depth reached)")
        return

    # Dict
    if isinstance(obj, dict):
        for k, v in obj.items():
            print(f"{indent}  Key: {k} | Type: {type(v)} | Value: {str(v)[:100]}")
            inspect_object(v, name=k, max_depth=max_depth, _depth=_depth + 1)

    # List
    elif isinstance(obj, list):
        print(f"{indent}  List length: {len(obj)}")
        for i, item in enumerate(obj[:3]):  # only first 3 items
            inspect_object(item, name=f"{name}[{i}]", max_depth=max_depth, _depth=_depth + 1)

    # Object (like WikiRate Answer)
    else:
        attrs = [attr for attr in dir(obj) if not attr.startswith("_")]

        for attr in attrs:
            try:
                value = getattr(obj, attr)
                print(f"{indent}  Attr: {attr} | Type: {type(value)} | Value: {str(value)[:100]}")
            except Exception as e:
                print(f"{indent}  Attr: {attr} | ERROR: {e}")

In [ ]:
for metric_id, years_data in raw_answers.items():
    for year, answers in years_data.items():
        print("\n" + "="*60)
        print(f"📊 Inspecting: {metric_id} | {year}")
        print("="*60)

        if not answers:
            print("⚠️ No answers found")
            continue

        # Inspecting first answer
        inspect_object(answers[0], name="answer", max_depth=2)

        break
    break


📊 Inspecting: GRI_305_1_a_Scope1_GHG | 2022
🔎 answer
Type: <class 'wikirate4py.models.AnswerItem'>
  Attr: check_requested | ERROR: 'AnswerItem' object has no attribute 'check_requested'
  Attr: checked_by | ERROR: 'AnswerItem' object has no attribute 'checked_by'
  Attr: comments | Type: <class 'str'> | Value: Page13 in 2020 annual report, P20 in 2020 sustainability report Chrisz[https://wikirate.org/Chrisz].
  Attr: company | Type: <class 'str'> | Value: Origin Energy
  Attr: extract_content | Type: <class 'function'> | Value: <function BaseEntity.extract_content at 0x787509674ae0>
  Attr: extract_id | Type: <class 'function'> | Value: <function BaseEntity.extract_id at 0x78750948c680>
  Attr: extract_name | Type: <class 'function'> | Value: <function BaseEntity.extract_name at 0x78750948c5e0>
  Attr: get_parameters | Type: <class 'method'> | Value: <bound method WikirateEntity.get_parameters of {'id': 7174977, 'metric': 'Global Reporting Initiativ
  Attr: id | Type: <class 'int'> |

In [ ]:
def normalize_answer(answer):
    data = answer.raw

    return {
        "company": data.get("company"),
        "metric": data.get("metric"),
        "year": data.get("year"),
        "value": float(data["value"]) if data.get("value") else None,
        "sources": data.get("sources", []),
        "url": data.get("url"),
        "comments": data.get("comments")
    }

# Get the normalized dictionary
normalized_data = normalize_answer(raw_answers['GRI_305_1_a_Scope1_GHG'][2022][0])

for key, value in normalized_data.items():
    print(f'"{key}": {value}')

"company": Origin Energy
"metric": Global Reporting Initiative+Direct greenhouse gas (GHG) emissions (Scope 1), GRI 305-1-a (formerly G4-EN15-a)
"year": 2022
"value": 22400000.0
"sources": ['Source-000104185']
"url": https://wikirate.org/Global_Reporting_Initiative+Direct_greenhouse_gas_GHG_emissions_Scope_1_GRI_305_1_a_formerly_G4_EN15_a+Origin_Energy+2022.json
"comments": Page13 in 2020 annual report, P20 in 2020 sustainability report Chrisz[https://wikirate.org/Chrisz].....2021-04-12 07:20:52 UTC According to p18 graph 5, 11/13*18648=15626.7692 Chrisz[https://wikirate.org/Chrisz].....2021-04-13 04:08:13 UTC


**3** **Pefroming valid answer filters**
- Applying base answer-level cleaning across all years and all records
- Handling null handling, Unknown, blanks, and duplicate logic as these effect signigficantly to the ML models through function ***is_valid_basic_answer***
- Taking measure to avoid duplicate company entries in the same year and for the same Metric

In [ ]:
def is_valid_basic_answer(answer):
    if answer.value is None:
        return False, "null_value"

    value_str = str(answer.value).strip()

    if value_str == "":
        return False, "empty_value"

    if value_str.lower() == "unknown":
        return False, "unknown_value"

    if not getattr(answer, "sources", None):
        return False, "missing_source_ids"

    return True, "kept"

filtered_answers = defaultdict(dict)
basic_quality_report = {}

for metric in METRICS:
    metric_id = metric["custom_id"]

    for year in YEARS:
        kept = []
        reject_counts = defaultdict(int)
        seen_company_year = set()

        for answer in raw_answers[metric_id][year]:
            key = (answer.company, answer.year)

            if key in seen_company_year:
                reject_counts["duplicate_company_year"] += 1
                continue

            is_valid, reason = is_valid_basic_answer(answer)
            if not is_valid:
                reject_counts[reason] += 1
                continue

            seen_company_year.add(key)
            kept.append(answer)

        filtered_answers[metric_id][year] = kept
        basic_quality_report[f"{metric_id}__{year}"] = {
            "raw_count": len(raw_answers[metric_id][year]),
            "kept_after_basic_filter": len(kept),
            "rejects": dict(reject_counts)
        }

for key, value in basic_quality_report.items():
    print(key, value)

GRI_305_1_a_Scope1_GHG__2022 {'raw_count': 15, 'kept_after_basic_filter': 12, 'rejects': {'unknown_value': 3}}
GRI_305_1_a_Scope1_GHG__2023 {'raw_count': 15, 'kept_after_basic_filter': 11, 'rejects': {'unknown_value': 4}}
GRI_102_8_a_Total_Employees__2022 {'raw_count': 15, 'kept_after_basic_filter': 12, 'rejects': {'unknown_value': 3}}
GRI_102_8_a_Total_Employees__2023 {'raw_count': 15, 'kept_after_basic_filter': 12, 'rejects': {'unknown_value': 3}}
GRI_102_8_a_Female_Employees__2022 {'raw_count': 15, 'kept_after_basic_filter': 7, 'rejects': {'unknown_value': 8}}
GRI_102_8_a_Female_Employees__2023 {'raw_count': 15, 'kept_after_basic_filter': 14, 'rejects': {'unknown_value': 1}}


**4** **Resolving and cleaning sources for kept answers.**
-  Resolving source IDs into source metadata for all kept answers across years
-  The data filtered before source expansion which can save time and number of  API calls made.

In [ ]:
'''
def fetch_source_objects(answer):
    source_objects = []

    for source_id in getattr(answer, "sources", []) or []:
        print(source_id)
        try:
            src = api.get_source(source_id)
            print("src....",src)
            source_objects.append(src)
            time.sleep(0.1)
        #except Exception:
            #continue

    return source_objects

def fetch_source_objects(answer):
    source_objects = []

    for source_id in getattr(answer, "sources", []) or []:
        print(source_id)

        retries = 0
        while retries < max_retries:
            try:
                src = api.get_source(source_id)
                print("src....", src)
                source_objects.append(src)
                time.sleep(0.2)  # slight delay
                break

            except TooManyRequestsException:
                wait_time = 2 ** retries
                print(f"⚠️ Rate limited. Retrying in {wait_time}s...")
                time.sleep(wait_time)
                retries += 1

            except Exception as e:
                print(f"❌ Failed for {source_id}: {e}")
                break

    return source_objects

'''

def fetch_source_objects(answer, max_retries=3):
    source_objects = []

    for source_id in getattr(answer, "sources", []) or []:
        print("Fetching:", source_id)

        retries = 0
        while retries < max_retries:
            try:
                src = api.get_source(source_id)
                #print("src....", src)
                source_objects.append(src)
                time.sleep(0.4)  # adding slight delay
                break

            except TooManyRequestsException:
                wait_time = 2 ** retries
                print(f"⚠️ Rate limited. Retrying in {wait_time}s...")
                time.sleep(wait_time)
                retries += 1

            except Exception as e:
                print(f"❌ Failed for {source_id}: {e}")
                break

    return source_objects

def build_file_metas_from_sources(source_objects):
    file_metas = []
    seen = set()

    for src in source_objects:                  #extracting file urls
        title = getattr(src, "title", None)
        file_url = getattr(src, "file_url", None)

        title = title.strip() if isinstance(title, str) else None
        file_url = file_url.strip() if isinstance(file_url, str) else None

        if not title and not file_url:
            continue

        key = (title, file_url)
        if key in seen:
            continue
        seen.add(key)

        file_metas.append({
            "file_name": title or "unknown_source_title",
            "file_url": file_url
        })

    return file_metas


answers_with_sources = defaultdict(dict)
source_quality_report = {}

for metric in METRICS:
    metric_id = metric["custom_id"]

    for year in YEARS:
        enriched_answers = []
        reject_counts = defaultdict(int)

        for answer in filtered_answers[metric_id][year]:
            print(year,metric_id,answer)
            source_objects = fetch_source_objects(answer)
            file_metas = build_file_metas_from_sources(source_objects)
            usable_file_metas = [fm for fm in file_metas if fm.get("file_url")]
            print(year,metric_id,usable_file_metas)

            if not usable_file_metas:
                reject_counts["missing_file_url"] += 1
                continue

            enriched_answers.append({
                "answer": answer,
                "file_metas": usable_file_metas
            })

        answers_with_sources[metric_id][year] = enriched_answers
        source_quality_report[f"{metric_id}__{year}"] = {
            "input_after_basic_filter": len(filtered_answers[metric_id][year]),
            "kept_after_source_cleaning": len(enriched_answers),
            "rejects": dict(reject_counts)
        }

for key, value in source_quality_report.items():
    print(key, value)

2022 GRI_305_1_a_Scope1_GHG {'id': 7174977, 'metric': 'Global Reporting Initiative+Direct greenhouse gas (GHG) emissions (Scope 1), GRI 305-1-a (formerly G4-EN15-a)', 'company': 'Origin Energy', 'value': '22400000', 'year': 2022, 'comments': 'Page13 in 2020 annual report, P20 in 2020 sustainability report Chrisz[https://wikirate.org/Chrisz].....2021-04-12 07:20:52 UTC According to p18 graph 5, 11/13*18648=15626.7692 Chrisz[https://wikirate.org/Chrisz].....2021-04-13 04:08:13 UTC', 'sources': ['Source-000104185'], 'checked_by': None, 'check_requested': None, 'url': 'https://wikirate.org/Global_Reporting_Initiative+Direct_greenhouse_gas_GHG_emissions_Scope_1_GRI_305_1_a_formerly_G4_EN15_a+Origin_Energy+2022'}
Fetching: Source-000104185
2022 GRI_305_1_a_Scope1_GHG [{'file_name': 'Origin Energy 2020 Annual Report', 'file_url': 'https://wikirate-production-storage.fra1.cdn.digitaloceanspaces.com/files/7174517/24701018.pdf'}]
2022 GRI_305_1_a_Scope1_GHG {'id': 13652934, 'metric': 'Global Rep

**5**.**Transforming into final records**
- Transforming the kept multi-year answers into the target schema
- Adding normalized numeric data in structured_data to make the better to understand by the ML model

In [ ]:
import re
from collections import defaultdict

def parse_numeric_value(value):
    if value is None:
        return None

    s = str(value).strip()
    if not s or s.lower() == "unknown":
        return None

    s = s.replace(",", "")
    match = re.search(r"-?\d+(?:\.\d+)?", s)
    if not match:
        return None

    num = float(match.group())
    return int(num) if num.is_integer() else num

dataset = []
transform_quality_report = {}

for metric in METRICS:
    metric_id = metric["custom_id"]

    for year in YEARS:
        records = []
        reject_counts = defaultdict(int)

        # Accessing the data stored in the previous enrichment step
        batch_data = answers_with_sources.get(metric_id, {}).get(year, [])

        for item in batch_data:
            answer = item["answer"]
            file_metas = item["file_metas"]

            numeric_value = parse_numeric_value(answer.value)
            if numeric_value is None:
                reject_counts["non_numeric_value"] += 1
                continue

            record = {
                "input": {
                    "question": metric["question"],
                    "custom_id": metric["custom_id"],
                    "data_type": "Metric",
                    "company": answer.company,
                    "file_metas": file_metas
                },
                "reference_output": {
                    "answer": str(answer.value).strip(),
                    "source_documents": file_metas
                },
                "structured_data": [
                    {
                        "unit": metric["unit"],
                        "value": numeric_value,
                        "time_period": str(answer.year)
                    }
                ],
                "metadata": {
                    "pillar": metric["pillar"],
                    "metric_name": metric["metric_name"],
                    "metric_designer": metric["metric_designer"]
                }
            }

            records.append(record)
            dataset.append(record)

        transform_quality_report[f"{metric_id}__{year}"] = {
            "input_after_source_cleaning": len(batch_data),
            "final_records": len(records),
            "rejects": dict(reject_counts)
        }

print("Total final dataset size:", len(dataset))
for key, value in transform_quality_report.items():
    print(key, value)

Total final dataset size: 68
GRI_305_1_a_Scope1_GHG__2022 {'input_after_source_cleaning': 12, 'final_records': 12, 'rejects': {}}
GRI_305_1_a_Scope1_GHG__2023 {'input_after_source_cleaning': 11, 'final_records': 11, 'rejects': {}}
GRI_102_8_a_Total_Employees__2022 {'input_after_source_cleaning': 12, 'final_records': 12, 'rejects': {}}
GRI_102_8_a_Total_Employees__2023 {'input_after_source_cleaning': 12, 'final_records': 12, 'rejects': {}}
GRI_102_8_a_Female_Employees__2022 {'input_after_source_cleaning': 7, 'final_records': 7, 'rejects': {}}
GRI_102_8_a_Female_Employees__2023 {'input_after_source_cleaning': 14, 'final_records': 14, 'rejects': {}}


**6**.**Overview of the Extracted data for insights**
 - Check for the Data-quality other traid of of the final dataset

In [ ]:
audit_rows = []

for metric in METRICS:
    metric_id = metric["custom_id"]

    for year in YEARS:
        basic = basic_quality_report.get(f"{metric_id}__{year}", {})
        source = source_quality_report.get(f"{metric_id}__{year}", {})
        final = transform_quality_report.get(f"{metric_id}__{year}", {})

        unique_companies = len({
            record["input"]["company"]
            for record in dataset
            if record["input"]["custom_id"] == metric_id
            and record["structured_data"][0]["time_period"] == str(year)
        })

        audit_rows.append({
            "metric_id": metric_id,
            "year": year,
            "raw_count": basic.get("raw_count", 0),
            "after_basic_filter": basic.get("kept_after_basic_filter", 0),
            "after_source_cleaning": source.get("kept_after_source_cleaning", 0),
            "final_records": final.get("final_records", 0),
            "unique_companies": unique_companies
        })

for row in audit_rows:
    print(row)

{'metric_id': 'GRI_305_1_a_Scope1_GHG', 'year': 2022, 'raw_count': 15, 'after_basic_filter': 12, 'after_source_cleaning': 12, 'final_records': 12, 'unique_companies': 12}
{'metric_id': 'GRI_305_1_a_Scope1_GHG', 'year': 2023, 'raw_count': 15, 'after_basic_filter': 11, 'after_source_cleaning': 11, 'final_records': 11, 'unique_companies': 11}
{'metric_id': 'GRI_102_8_a_Total_Employees', 'year': 2022, 'raw_count': 15, 'after_basic_filter': 12, 'after_source_cleaning': 12, 'final_records': 12, 'unique_companies': 12}
{'metric_id': 'GRI_102_8_a_Total_Employees', 'year': 2023, 'raw_count': 15, 'after_basic_filter': 12, 'after_source_cleaning': 12, 'final_records': 12, 'unique_companies': 12}
{'metric_id': 'GRI_102_8_a_Female_Employees', 'year': 2022, 'raw_count': 15, 'after_basic_filter': 7, 'after_source_cleaning': 7, 'final_records': 7, 'unique_companies': 7}
{'metric_id': 'GRI_102_8_a_Female_Employees', 'year': 2023, 'raw_count': 15, 'after_basic_filter': 14, 'after_source_cleaning': 14, '

**7. Exporting the dataset**
- Exporting the final dataset list into a JSON file

In [ ]:
import json
from pathlib import Path

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "wikirate_esg_dataset.json"

print("Number of records in dataset:", len(dataset))

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        dataset,
        f,
        indent=2,
        ensure_ascii=False
    )

print("JSON dataset exported to:", OUTPUT_PATH.resolve())

Number of records in dataset: 68
JSON dataset exported to: /content/output/wikirate_esg_dataset.json
